# IQ or spectrogram for modulation recognition?

This notebook builds intuition for choosing a model input representation for **modulation recognition (modrec)**. It uses small synthetic examples rather than training a production classifier, so it runs quickly on a CPU.

**Short answer:** start with IQ when phase and symbol transitions carry class information; start with a spectrogram when time-frequency occupancy and robustness to phase/timing nuisance matter more. Validate both under the impairments expected at deployment.

## What each representation gives the model

| Property | IQ samples | Magnitude/power spectrogram |
|---|---|---|
| Preserves instantaneous phase | Yes | No |
| Preserves fine symbol timing | Yes | Limited by STFT window/hop |
| Exposes frequency-vs-time structure | Must be learned | Explicit |
| Naturally invariant to global phase | No (unless learned/augmented) | Yes |
| Typical tensor | `2 x N` real/imag | `C x F x T` image-like |
| Good starting point | PSK/QAM distinctions, coherent structure | FSK/chirps/bursts, frequency hopping, image backbones |

A representation does not determine accuracy by itself. Architecture, receptive field, preprocessing, augmentation, SNR, channel model, and dataset coverage can dominate the result.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
plt.rcParams.update({"figure.figsize": (10, 3.2), "image.aspect": "auto"})

## Generate simple baseband signals

These signals use rectangular pulses to keep the code transparent. Production experiments should use realistic pulse shaping, sample-rate offsets, multipath, fading, nonlinearities, and receiver filtering.

In [ ]:
def constellation(name):
    if name == "BPSK":
        points = np.array([-1, 1], dtype=complex)
    elif name == "QPSK":
        points = np.exp(1j * (np.pi / 4 + np.arange(4) * np.pi / 2))
    elif name == "16-QAM":
        levels = np.array([-3, -1, 1, 3])
        points = (levels[:, None] + 1j * levels[None, :]).ravel()
    else:
        raise ValueError(f"Unknown modulation: {name}")
    return points / np.sqrt(np.mean(np.abs(points) ** 2))

def make_signal(name, n_symbols=128, sps=8, snr_db=18, phase=0.0, cfo=0.0):
    points = constellation(name)
    symbols = points[rng.integers(0, len(points), n_symbols)]
    clean = np.repeat(symbols, sps)
    t = np.arange(clean.size)
    clean = clean * np.exp(1j * (phase + 2 * np.pi * cfo * t))
    noise_power = np.mean(np.abs(clean) ** 2) / (10 ** (snr_db / 10))
    noise = np.sqrt(noise_power / 2) * (rng.normal(size=t.size) + 1j * rng.normal(size=t.size))
    return clean + noise

def stft_power(x, n_fft=64, hop=16):
    if len(x) < n_fft:
        raise ValueError("x must contain at least n_fft samples")
    starts = np.arange(0, len(x) - n_fft + 1, hop)
    frames = np.stack([x[start:start + n_fft] for start in starts])
    spectrum = np.fft.fftshift(np.fft.fft(frames * np.hanning(n_fft), axis=1), axes=1)
    return 10 * np.log10(np.abs(spectrum.T) ** 2 + 1e-12)

In [ ]:
names = ["BPSK", "QPSK", "16-QAM"]
signals = {name: make_signal(name) for name in names}
fig, axes = plt.subplots(len(names), 3, figsize=(13, 9))
for row, name in enumerate(names):
    x = signals[name]
    axes[row, 0].plot(x.real[:160], label="I", lw=1)
    axes[row, 0].plot(x.imag[:160], label="Q", lw=1, alpha=.8)
    axes[row, 0].set_ylabel(name)
    axes[row, 1].scatter(x.real[::8], x.imag[::8], s=8, alpha=.45)
    axes[row, 1].set_aspect("equal")
    axes[row, 2].imshow(stft_power(x), origin="lower", cmap="magma")
for ax, title in zip(axes[0], ["IQ versus sample", "Symbol-rate IQ", "Power spectrogram"]):
    ax.set_title(title)
axes[0, 0].legend(loc="upper right")
plt.tight_layout()

The constellation view makes IQ's advantage clear: BPSK, QPSK, and 16-QAM differ directly in phase/amplitude geometry. Their power spectrograms can look quite similar because magnitude STFT discards much of that geometry. Thus a magnitude spectrogram may create an **information bottleneck** for closely related linear modulations.

## A useful spectrogram invariance is also information loss

Multiplying all samples by a constant phase rotation changes IQ coordinates. It leaves STFT magnitude essentially unchanged. That is helpful when absolute carrier phase is a nuisance—but harmful if the discarded phase relationships distinguish labels.

In [ ]:
x = signals["QPSK"]
x_rotated = x * np.exp(1j * 1.1)
iq_change = np.linalg.norm(x - x_rotated) / np.linalg.norm(x)
spec_change = np.linalg.norm(stft_power(x) - stft_power(x_rotated)) / np.linalg.norm(stft_power(x))
print(f"Relative IQ change:          {iq_change:.3f}")
print(f"Relative spectrogram change: {spec_change:.3e}")

fig, axes = plt.subplots(1, 2)
axes[0].scatter(x.real[::8], x.imag[::8], s=10, label="original")
axes[0].scatter(x_rotated.real[::8], x_rotated.imag[::8], s=10, label="rotated")
axes[0].set_aspect("equal"); axes[0].legend(); axes[0].set_title("IQ changes")
axes[1].imshow(stft_power(x_rotated) - stft_power(x), origin="lower", cmap="coolwarm")
axes[1].set_title("Spectrogram difference (dB)")
plt.tight_layout()

## The STFT resolution trade-off

A short window localizes transitions in time but has coarse frequency resolution. A long window resolves frequencies but smears short events. `n_fft` and hop length are therefore model hyperparameters, not cosmetic plotting choices.

In [ ]:
x = make_signal("QPSK", n_symbols=256, cfo=0.006)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, n_fft in zip(axes, [32, 64, 256]):
    hop = n_fft // 4
    image = stft_power(x, n_fft=n_fft, hop=hop)
    ax.imshow(image, origin="lower", cmap="magma")
    ax.set_title(f"n_fft={n_fft}, shape={image.shape}")
plt.tight_layout()

## Input size and compute implications

For `N` complex samples, a real-valued IQ tensor has `2N` values. A spectrogram has roughly `F × T` values per channel and also incurs STFT preprocessing. Overlap, FFT size, cropping, and whether phase is retained determine its actual footprint. The comparison below counts values, not model FLOPs; a 1-D and 2-D network can have very different costs even at equal input size.

In [ ]:
n = len(signals["QPSK"])
print(f"IQ (I and Q): {2 * n:,} float values")
for n_fft, hop in [(64, 16), (128, 32), (256, 64)]:
    shape = stft_power(signals["QPSK"], n_fft, hop).shape
    print(f"Spectrogram n_fft={n_fft:3}, hop={hop:2}: {shape[0] * shape[1]:,} values, shape={shape}")

## Practical decision guide

Choose **IQ first** when:

- labels differ through constellation geometry, phase trajectories, or symbol transitions (for example PSK order, QAM order, OQPSK/MSK variants);
- the receiver preserves trustworthy complex samples and fine timing;
- you can train with realistic phase, frequency, timing, gain, and channel augmentation;
- low-latency streaming and avoiding an STFT are important.

Choose a **spectrogram first** when:

- class evidence is primarily time-frequency shape (FSK tones, chirps, hopping, bandwidth, burst cadence);
- global phase is nuisance and time/frequency translation robustness is valuable;
- you want to exploit mature 2-D CNN or vision backbones and pretrained features;
- interpretability for operators through time-frequency plots matters.

Try **both or fuse them** when:

- the label set mixes linear modulations with FSK/chirp/burst families;
- deployment impairments are uncertain;
- errors from the two representations are complementary.

Alternatives include complex STFT (retains phase), amplitude/phase channels, cyclic features, or separate IQ and spectrogram branches.

## How to make the choice experimentally

1. Use identical train/validation/test signal identities and splits for both representations to prevent leakage.
2. Match observation duration and approximately match model capacity and tuning effort.
3. Sweep SNR and deployment impairments separately: carrier offset, sample-rate offset, timing, multipath, fading, clipping, and interference.
4. Report per-class confusion matrices and accuracy versus SNR—not only aggregate accuracy.
5. Measure latency, memory, calibration, and out-of-distribution behavior on target hardware.
6. Inspect whether spectrogram-confused classes differ mainly by phase; inspect whether IQ-confused classes differ mainly by time-frequency occupancy.

> **Rule of thumb, not a verdict:** IQ preserves more of the received signal, while a magnitude spectrogram supplies a useful inductive bias by deliberately discarding phase and reorganizing energy in time and frequency. The best representation is the one whose bias matches the label information and deployment nuisances.